<a href="https://colab.research.google.com/github/Binay1007/AI-resume-screener/blob/main/Resume_Screnner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import pickle
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

df = pd.read_csv("resume_text_category.csv")

df[['resume_text','category']] = df['resume_text,category'].str.split(',', n=1, expand=True)
df = df.drop(columns=['resume_text,category'])

print("Dataset Loaded")
print(df.head())

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df["resume_text"] = df["resume_text"].apply(clean_text)

texts = df["resume_text"]
labels = df["category"]

encoder = LabelEncoder()
y = encoder.fit_transform(labels)

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen=50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = Sequential()

model.add(Embedding(5000, 64, input_length=50))
model.add(LSTM(64))
model.add(Dense(32, activation="relu"))
model.add(Dense(len(set(y)), activation="softmax"))

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(X_train, y_train, epochs=20, batch_size=4)

loss, acc = model.evaluate(X_test, y_test)

print("Model Accuracy:", acc)

model.save("resume_model.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Model Saved")

def predict_resume(text):

    text = clean_text(text)

    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=50)

    prediction = model.predict(padded)

    label = encoder.inverse_transform([prediction.argmax()])

    return label[0]

resume = input("\nPaste Resume Text:\n")

result = predict_resume(resume)

print("\nPredicted Job Category:", result)

FileNotFoundError: [Errno 2] No such file or directory: 'resume_text_category.csv'